# NB-2: HotPotQA + MuSiQue Multi-Hop Benchmarks
**Publication blocker PB-13 (L3 hop isolation)**

Runs two multi-hop QA benchmarks:
- **HotPotQA** — 2-hop retrieval over Wikipedia-style passages
- **MuSiQue** — multi-hop reasoning (run twice: with L3 and without L3)

Comparing MuSiQue with/without L3 isolates the knowledge-graph hop contribution.

**Datasets:** hotpotqa_dev.json (7405 Q, we use 50), musique_dev.jsonl (200 Q)

**Time estimate:** ~15 min per run

## Step 1 — Install & clone

In [ ]:
import os, subprocess, sys

subprocess.run([sys.executable, '-m', 'pip', 'install', '-q',
    'sentence-transformers', 'hnswlib', 'python-dotenv', 'groq', 'requests'], check=True)

REPO = 'https://github.com/Lamaq-Mujpurwala/CSAM-IPD-HALH.git'
REPO_DIR = '/kaggle/working/CSAM-IPD-HALH' if os.path.exists('/kaggle') else '/content/CSAM-IPD-HALH'

if os.path.exists(REPO_DIR):
    print('Pulling latest changes from main...')
    result = subprocess.run(['git', '-C', REPO_DIR, 'pull', 'origin', 'main'],
                          capture_output=True, text=True)
    print(result.stdout)
    if result.returncode != 0:
        print('⚠ Pull warning:', result.stderr)
else:
    subprocess.run(['git', 'clone', '--depth', '1', REPO, REPO_DIR], check=True)

os.chdir(REPO_DIR)
sys.path.insert(0, os.path.join(REPO_DIR, 'csam_project'))
print(f'✓ Ready in {os.getcwd()}')

# Show latest commit
commit_result = subprocess.run(['git', 'log', '-1', '--oneline'], 
                              capture_output=True, text=True, cwd=REPO_DIR)
print(f'✓ Latest commit: {commit_result.stdout.strip()}')

## Step 2 — API key
**Kaggle:** Notebook → Settings → Secrets → `GROQ_API_KEY`

**Colab:** Left sidebar key icon → `GROQ_API_KEY`

In [ ]:
import os

def load_api_key():
    try:
        from kaggle_secrets import UserSecretsClient
        key = UserSecretsClient().get_secret('GROQ_API_KEY')
        if key: os.environ['GROQ_API_KEY'] = key; return 'kaggle'
    except Exception: pass
    try:
        from google.colab import userdata
        key = userdata.get('GROQ_API_KEY')
        if key: os.environ['GROQ_API_KEY'] = key; return 'colab'
    except Exception: pass
    if os.environ.get('GROQ_API_KEY'): return 'env'
    raise RuntimeError('Add GROQ_API_KEY to Kaggle/Colab Secrets')

print(f'API key loaded from: {load_api_key()}')
with open('.env', 'w') as f:
    f.write(f"GROQ_API_KEY={os.environ['GROQ_API_KEY']}\n")

In [ ]:
# ============================================================================
# VALIDATION: Test Semantic Similarity Function
# Verifies that the semantic_f1 metric is correctly loaded and functional
# ============================================================================

print("\n" + "="*70)
print("VALIDATING SEMANTIC SIMILARITY METRIC")
print("="*70)

try:
    from benchmarks import metrics as metrics_module
    print("\n✓ Imported metrics module successfully")
    
    # Check if semantic_f1 function exists
    if hasattr(metrics_module, 'semantic_f1'):
        print("✓ semantic_f1 function found in metrics module")
    else:
        raise AttributeError("semantic_f1 function NOT found")
    
    # Check if cosine_sim function exists
    if hasattr(metrics_module, 'cosine_sim'):
        print("✓ cosine_sim function found in metrics module")
    else:
        raise AttributeError("cosine_sim function NOT found")
    
    # Test with EmbeddingService
    from csam_core.services.embedding import EmbeddingService
    print("✓ Imported EmbeddingService")
    
    embedding_service = EmbeddingService()
    print("✓ EmbeddingService initialized")
    
    # Quick sanity test
    test_pred = "The answer is flying"
    test_truth = "flying over buildings"
    
    sem_score = metrics_module.semantic_f1(test_pred, test_truth, embedding_service.encode)
    print(f"✓ semantic_f1 test: '{test_pred}' vs '{test_truth}'")
    print(f"  Semantic similarity: {sem_score:.4f}")
    
    if 0.0 <= sem_score <= 1.0:
        print("✓ Semantic similarity score is in valid range [0.0, 1.0]")
    else:
        raise ValueError(f"Score out of range: {sem_score}")
    
    print("\n" + "="*70)
    print("SUCCESS: All semantic similarity components loaded correctly!")
    print("="*70)
    print("\nN2 Benchmarks will now include semantic similarity metrics in results.")
    
except Exception as e:
    print(f"\n❌ ERROR: {type(e).__name__}: {e}")
    print("\nTroubleshooting:")
    print("  1. Check that git pull completed successfully in Step 1")
    print("  2. Verify metrics.py has semantic_f1 and cosine_sim functions")
    print("  3. Ensure GROQ_API_KEY is properly set in Step 2")
    import traceback
    traceback.print_exc()

## Step 3 — Configure

In [ ]:
import os

PROVIDER   = 'groq'
MODEL      = 'llama-3.1-8b-instant'
SEED       = 42
CHECKPOINT = '/kaggle/working' if os.path.exists('/kaggle') else '/content'

# HotPotQA: how many questions (max 7405, use 50 for quick, 200 for paper)
HOTPOT_Q   = 50

# Use absolute paths to datasets
HOTPOT_DS  = os.path.join(REPO_DIR, 'csam_project', 'benchmarks', 'data', 'hotpotqa_dev.json')
MUSIQUE_DS = os.path.join(REPO_DIR, 'csam_project', 'benchmarks', 'data', 'musique_dev.jsonl')

# Verify datasets exist
for label, path in [('HotPotQA', HOTPOT_DS), ('MuSiQue', MUSIQUE_DS)]:
    exists = os.path.exists(path)
    print(f'{label:<12} {path}')
    print(f'  Exists: {exists}')
    if not exists and os.path.exists(os.path.dirname(path)):
        print(f'  Files in dir: {os.listdir(os.path.dirname(path))}')

safe_model = MODEL.replace('/', '_')
OUT_HOTPOT       = f'csam_project/benchmarks/results_hotpotqa_{PROVIDER}_{safe_model}.json'
OUT_MUSIQUE      = f'csam_project/benchmarks/results_musique_{PROVIDER}_{safe_model}.json'
OUT_MUSIQUE_NOL3 = f'csam_project/benchmarks/results_musique_{PROVIDER}_{safe_model}_nol3.json'

print(f'\nModel: {MODEL} | HotPotQA Q: {HOTPOT_Q} | MuSiQue: all 200')

## Step 4 — Run HotPotQA

In [ ]:
import subprocess, sys, os
os.chdir(REPO_DIR)

cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_hotpotqa',
    '--provider', PROVIDER,
    '--model', MODEL,
    '--questions', str(HOTPOT_Q),
    '--dataset', HOTPOT_DS,
    '--seed', str(SEED),
    '--checkpoint-dir', CHECKPOINT,
]
print('Running HotPotQA...')
result = subprocess.run(cmd, capture_output=False, text=True)
print('HotPotQA done' if result.returncode == 0 else 'HotPotQA FAILED')

## Step 5 — Run MuSiQue (with L3)
Full CSAM system — L3 knowledge graph enables multi-hop retrieval.

In [ ]:
cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_musique',
    '--provider', PROVIDER,
    '--model', MODEL,
    '--seed', str(SEED),
    '--checkpoint-dir', CHECKPOINT,
]
print('Running MuSiQue WITH L3...')
result = subprocess.run(cmd, capture_output=False, text=True)
print('MuSiQue+L3 done' if result.returncode == 0 else 'MuSiQue+L3 FAILED')

## Step 6 — Run MuSiQue (WITHOUT L3)
Ablation: same setup but L3 knowledge graph is disabled.
The F1 delta vs the L3 run quantifies the graph hop contribution (PB-13).

In [ ]:
cmd = [
    sys.executable, '-m', 'csam_project.benchmarks.benchmark_musique',
    '--provider', PROVIDER,
    '--model', MODEL,
    '--seed', str(SEED),
    '--checkpoint-dir', CHECKPOINT,
    '--no-l3',   # ← disables L3 knowledge graph
]
print('Running MuSiQue WITHOUT L3...')
result = subprocess.run(cmd, capture_output=False, text=True)
print('MuSiQue-noL3 done' if result.returncode == 0 else 'MuSiQue-noL3 FAILED')

## Step 7 — Results summary

In [ ]:
import json, os

print('=' * 60)
print('RESULTS SUMMARY')
print('=' * 60)

def show(label, path):
    if not os.path.exists(path):
        print(f'{label:<30} NOT FOUND ({path})')
        return
    with open(path) as f:
        d = json.load(f)
    f1 = d.get('micro_f1') or d.get('avg_f1') or d.get('overall_f1', 0)
    n  = d.get('num_questions') or len(d.get('per_conversation', []))
    print(f'{label:<30} F1={f1:.4f}  n={n}')

show('HotPotQA (CSAM)',   OUT_HOTPOT)
show('MuSiQue (with L3)', OUT_MUSIQUE)
show('MuSiQue (no L3)',   OUT_MUSIQUE_NOL3)

# L3 hop delta
if os.path.exists(OUT_MUSIQUE) and os.path.exists(OUT_MUSIQUE_NOL3):
    with open(OUT_MUSIQUE) as f: m_l3 = json.load(f)
    with open(OUT_MUSIQUE_NOL3) as f: m_nl3 = json.load(f)
    f1_l3  = m_l3.get('micro_f1') or m_l3.get('avg_f1', 0)
    f1_nl3 = m_nl3.get('micro_f1') or m_nl3.get('avg_f1', 0)
    print(f'\nL3 hop contribution: {f1_l3 - f1_nl3:+.4f} F1 points')

## Step 8 — Save output files

In [ ]:
import shutil

files_to_save = [OUT_HOTPOT, OUT_MUSIQUE, OUT_MUSIQUE_NOL3]

if os.path.exists('/kaggle'):
    for fp in files_to_save:
        if os.path.exists(fp):
            dest = os.path.join('/kaggle/working', os.path.basename(fp))
            shutil.copy(fp, dest)
            print(f'Kaggle output: {dest}')
else:
    try:
        from google.colab import files
        for fp in files_to_save:
            if os.path.exists(fp): files.download(fp); print(f'Downloaded: {fp}')
    except ImportError:
        print('Files at:', [fp for fp in files_to_save if os.path.exists(fp)])